# Multi-Factor Equity Screener
A quantitative equity screener that ranks S&P 500 stocks across five factor families — **Value**, **Quality**, **Momentum**, **Low Volatility**, and **Growth** — using live data from Yahoo Finance. Each stock receives a composite Z-score ranking. Results are visualized via interactive charts and a styled factor heatmap, and the top-ranked names are surfaced as a final screener output.

In [ ]:
# ── Installs & Imports ────────────────────────────────────────────────────────
!pip install yfinance pandas-datareader plotly kaleido --quiet

import warnings
warnings.filterwarnings("ignore")

import yfinance as yf
import pandas as pd
import numpy as np
import requests
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from datetime import datetime, timedelta
from io import StringIO
import time

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 30)
SEED = 42

## 1. Universe Construction
Pull the current S&P 500 constituent list from Wikipedia, then filter to a working subset for data reliability.

In [ ]:
# ── Universe: S&P 500 tickers ─────────────────────────────────────────────────
import requests, pandas as pd

def fetch_sp500_tickers() -> pd.DataFrame:
    # Primary: fetch from GitHub-hosted CSV (reliable in Colab)
    try:
        url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"
        df = pd.read_csv(url)
        df.columns = [c.strip() for c in df.columns]
        df = df.rename(columns={
            "Symbol": "ticker",
            "Name": "name",
            "Sector": "sector",
        })
        df["ticker"] = df["ticker"].str.replace(".", "-", regex=False)
        df["sub_industry"] = ""
        print(f"✅ Loaded {len(df)} tickers from GitHub CSV")
        return df[["ticker", "name", "sector", "sub_industry"]].reset_index(drop=True)

    except Exception as e:
        print(f"GitHub CSV failed ({e}), falling back to hardcoded sector map …")

        # Fallback: manually curated list of ~100 liquid names across all sectors
        data = [
            ("AAPL","Apple Inc","Information Technology"),
            ("MSFT","Microsoft Corp","Information Technology"),
            ("NVDA","NVIDIA Corp","Information Technology"),
            ("GOOGL","Alphabet Inc","Communication Services"),
            ("META","Meta Platforms","Communication Services"),
            ("AMZN","Amazon.com Inc","Consumer Discretionary"),
            ("TSLA","Tesla Inc","Consumer Discretionary"),
            ("JPM","JPMorgan Chase","Financials"),
            ("BAC","Bank of America","Financials"),
            ("GS","Goldman Sachs","Financials"),
            ("MS","Morgan Stanley","Financials"),
            ("V","Visa Inc","Financials"),
            ("MA","Mastercard Inc","Financials"),
            ("UNH","UnitedHealth Group","Health Care"),
            ("JNJ","Johnson & Johnson","Health Care"),
            ("LLY","Eli Lilly","Health Care"),
            ("ABBV","AbbVie Inc","Health Care"),
            ("MRK","Merck & Co","Health Care"),
            ("PFE","Pfizer Inc","Health Care"),
            ("XOM","Exxon Mobil","Energy"),
            ("CVX","Chevron Corp","Energy"),
            ("COP","ConocoPhillips","Energy"),
            ("NEE","NextEra Energy","Utilities"),
            ("DUK","Duke Energy","Utilities"),
            ("SO","Southern Co","Utilities"),
            ("PG","Procter & Gamble","Consumer Staples"),
            ("KO","Coca-Cola Co","Consumer Staples"),
            ("PEP","PepsiCo Inc","Consumer Staples"),
            ("WMT","Walmart Inc","Consumer Staples"),
            ("COST","Costco Wholesale","Consumer Staples"),
            ("CAT","Caterpillar Inc","Industrials"),
            ("DE","Deere & Co","Industrials"),
            ("RTX","RTX Corp","Industrials"),
            ("HON","Honeywell Intl","Industrials"),
            ("UPS","United Parcel Service","Industrials"),
            ("BA","Boeing Co","Industrials"),
            ("LMT","Lockheed Martin","Industrials"),
            ("AMT","American Tower","Real Estate"),
            ("PLD","Prologis Inc","Real Estate"),
            ("EQIX","Equinix Inc","Real Estate"),
            ("BLK","BlackRock Inc","Financials"),
            ("SCHW","Charles Schwab","Financials"),
            ("SPGI","S&P Global Inc","Financials"),
            ("AMD","Advanced Micro Devices","Information Technology"),
            ("INTC","Intel Corp","Information Technology"),
            ("CRM","Salesforce Inc","Information Technology"),
            ("ORCL","Oracle Corp","Information Technology"),
            ("ADBE","Adobe Inc","Information Technology"),
            ("NOW","ServiceNow Inc","Information Technology"),
            ("QCOM","Qualcomm Inc","Information Technology"),
            ("TXN","Texas Instruments","Information Technology"),
            ("AMAT","Applied Materials","Information Technology"),
            ("MU","Micron Technology","Information Technology"),
            ("NFLX","Netflix Inc","Communication Services"),
            ("DIS","Walt Disney Co","Communication Services"),
            ("CMCSA","Comcast Corp","Communication Services"),
            ("T","AT&T Inc","Communication Services"),
            ("VZ","Verizon Communications","Communication Services"),
            ("HD","Home Depot Inc","Consumer Discretionary"),
            ("MCD","McDonald's Corp","Consumer Discretionary"),
            ("NKE","Nike Inc","Consumer Discretionary"),
            ("SBUX","Starbucks Corp","Consumer Discretionary"),
            ("TJX","TJX Companies","Consumer Discretionary"),
            ("LOW","Lowe's Companies","Consumer Discretionary"),
            ("F","Ford Motor Co","Consumer Discretionary"),
            ("GM","General Motors","Consumer Discretionary"),
            ("CVS","CVS Health","Health Care"),
            ("BMY","Bristol-Myers Squibb","Health Care"),
            ("AMGN","Amgen Inc","Health Care"),
            ("GILD","Gilead Sciences","Health Care"),
            ("SLB","SLB (Schlumberger)","Energy"),
            ("EOG","EOG Resources","Energy"),
            ("MPC","Marathon Petroleum","Energy"),
            ("PSX","Phillips 66","Energy"),
            ("AEP","American Electric Power","Utilities"),
            ("EXC","Exelon Corp","Utilities"),
            ("WM","Waste Management","Industrials"),
            ("GE","GE Aerospace","Industrials"),
            ("MMM","3M Co","Industrials"),
            ("FDX","FedEx Corp","Industrials"),
            ("CSX","CSX Corp","Industrials"),
            ("NSC","Norfolk Southern","Industrials"),
            ("ECL","Ecolab Inc","Materials"),
            ("LIN","Linde PLC","Materials"),
            ("APD","Air Products","Materials"),
            ("NEM","Newmont Corp","Materials"),
            ("FCX","Freeport-McMoRan","Materials"),
            ("DOW","Dow Inc","Materials"),
            ("SPG","Simon Property Group","Real Estate"),
            ("O","Realty Income Corp","Real Estate"),
            ("WELL","Welltower Inc","Real Estate"),
            ("CB","Chubb Ltd","Financials"),
            ("MMC","Marsh & McLennan","Financials"),
            ("AXP","American Express","Financials"),
            ("WFC","Wells Fargo","Financials"),
            ("C","Citigroup Inc","Financials"),
            ("USB","US Bancorp","Financials"),
            ("PNC","PNC Financial","Financials"),
        ]
        df = pd.DataFrame(data, columns=["ticker","name","sector"])
        df["sub_industry"] = ""
        print(f"✅ Loaded {len(df)} tickers from hardcoded fallback")
        return df[["ticker","name","sector","sub_industry"]].reset_index(drop=True)

sp500 = fetch_sp500_tickers()
print(f"Universe: {len(sp500)} stocks across {sp500['sector'].nunique()} sectors")
sp500.head()

## 2. Raw Price & Fundamental Data Pipeline
Download 1 year of adjusted closing prices for every ticker, then pull fundamental metrics via `yfinance` info dictionaries. This cell handles rate-limit back-off and missing data gracefully.

In [ ]:
# ── Price Download (1 year) ───────────────────────────────────────────────────
END   = datetime.today()
START = END - timedelta(days=365)

TICKERS = sp500["ticker"].tolist()[:30]

print("Downloading price history (batched) …")
raw_prices = yf.download(
    TICKERS,
    start=START.strftime("%Y-%m-%d"),
    end=END.strftime("%Y-%m-%d"),
    auto_adjust=True,
    progress=True,
    threads=True,
)["Close"]

# Drop tickers with > 20 % missing sessions
threshold = 0.80 * len(raw_prices)
prices = raw_prices.dropna(axis=1, thresh=int(threshold)).ffill().bfill()

valid_tickers = prices.columns.tolist()
sp500 = sp500[sp500["ticker"].isin(valid_tickers)].reset_index(drop=True)
print(f"\nValid tickers after price filter: {len(valid_tickers)}")

In [ ]:
# ── Fundamental Data via yfinance .info ──────────────────────────────────────
FUNDAMENTAL_FIELDS = {
    "trailingPE":          "pe_ratio",
    "priceToBook":         "pb_ratio",
    "enterpriseToEbitda":  "ev_ebitda",
    "trailingEps":         "eps_ttm",
    "revenueGrowth":       "revenue_growth_yoy",
    "earningsGrowth":      "earnings_growth_yoy",
    "returnOnEquity":      "roe",
    "returnOnAssets":      "roa",
    "debtToEquity":        "debt_to_equity",
    "currentRatio":        "current_ratio",
    "profitMargins":       "net_margin",
    "operatingMargins":    "operating_margin",
    "freeCashflow":        "fcf",
    "marketCap":           "market_cap",
    "forwardPE":           "forward_pe",
}

def fetch_fundamentals(tickers: list, pause: float = 0.05) -> pd.DataFrame:
    records = []
    for i, tkr in enumerate(tickers):
        try:
            info = yf.Ticker(tkr).info
            row = {"ticker": tkr}
            for yf_key, col in FUNDAMENTAL_FIELDS.items():
                row[col] = info.get(yf_key, np.nan)
            records.append(row)
        except Exception:
            records.append({"ticker": tkr})
        if i % 50 == 0 and i > 0:
            print(f"  {i}/{len(tickers)} fetched …")
            time.sleep(pause)
    return pd.DataFrame(records).set_index("ticker")

print("Fetching fundamental data (this takes ~3–5 min for 500 stocks) …")
fundamentals = fetch_fundamentals(valid_tickers)
print(f"\nFundamentals shape: {fundamentals.shape}")
print(f"Coverage (non-NaN %) per field:")
print((fundamentals.notna().mean() * 100).round(1).to_string())

## 3. Factor Construction
Each factor family is built from raw fundamentals and price data, then **cross-sectionally Z-scored** so they are comparable. Winsorization at the 2.5th/97.5th percentile prevents outliers from dominating.

In [ ]:
# ── Helper: Winsorize + Z-Score ───────────────────────────────────────────────
def winsorize_zscore(series: pd.Series, lo: float = 0.025, hi: float = 0.975) -> pd.Series:
    """Winsorize at [lo, hi] quantiles, then cross-sectionally Z-score."""
    s = series.copy()
    lower, upper = s.quantile(lo), s.quantile(hi)
    s = s.clip(lower, upper)
    mu, sigma = s.mean(), s.std()
    if sigma < 1e-10:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mu) / sigma

In [ ]:
# ── Price-Derived Features ────────────────────────────────────────────────────
returns_daily = prices.pct_change()

n = len(prices)  # actual number of trading days available

# ── Momentum: 12-1 month ──────────────────────────────────────────────────────
# Needs ~252 days back and ~21 days skip; fall back gracefully if shorter
lookback   = min(251, n - 2)   # index of ~1 year ago
skip       = min(20,  n - 2)   # index of ~1 month ago (short-term reversal skip)

if n > skip + 1:
    price_start = prices.iloc[-(lookback + 1)]   # ~12 months ago
    price_skip  = prices.shift(skip).iloc[-1]    # price as of 1 month ago
    mom_12_1    = (price_skip / price_start - 1).rename("momentum_12_1")
else:
    mom_12_1 = pd.Series(np.nan, index=prices.columns, name="momentum_12_1")

# ── 1-month short-term reversal ───────────────────────────────────────────────
if n > skip + 1:
    reversal_1m = (prices.iloc[-1] / prices.shift(skip).iloc[-1] - 1).rename("reversal_1m")
else:
    reversal_1m = pd.Series(np.nan, index=prices.columns, name="reversal_1m")

# ── Realised volatility (annualised, up to 90-day window) ────────────────────
vol_window  = min(90, n - 1)
vol_90d     = (returns_daily.tail(vol_window).std() * np.sqrt(252)).rename("vol_90d")

# ── Sharpe proxy ──────────────────────────────────────────────────────────────
sharpe_proxy = (mom_12_1 / vol_90d.replace(0, np.nan)).rename("sharpe_proxy")

# ── Max drawdown over available history ──────────────────────────────────────
def max_drawdown(price_series: pd.Series) -> float:
    roll_max = price_series.cummax()
    dd = (price_series - roll_max) / roll_max.replace(0, np.nan)
    return dd.min()

mdd = prices.apply(max_drawdown).rename("max_drawdown")

price_features = pd.concat([mom_12_1, reversal_1m, vol_90d, sharpe_proxy, mdd], axis=1)
price_features.index.name = "ticker"

print(f"Trading days available : {n}")
print(f"Momentum lookback used : {lookback} days")
print(f"Reversal skip used     : {skip} days")
print(f"Vol window used        : {vol_window} days")
print(f"Price features shape   : {price_features.shape}")
price_features.describe().round(4)

In [ ]:
# ── Merge All Raw Features ────────────────────────────────────────────────────
raw = fundamentals.join(price_features, how="inner")
raw = raw.join(sp500.set_index("ticker")[["sector", "name"]], how="left")
print(f"Combined feature matrix: {raw.shape}")
raw.head(3)

In [ ]:
# ── Factor Scoring ─────────────────────────────────────────────────────────────

factors = pd.DataFrame(index=raw.index)

# ── VALUE FACTOR ──────────────────────────────────────────────────────────────
# Lower PE/PB/EV-EBITDA = better value → negate
factors["val_pe"]       = winsorize_zscore(-raw["pe_ratio"].dropna().reindex(raw.index))
factors["val_pb"]       = winsorize_zscore(-raw["pb_ratio"].dropna().reindex(raw.index))
factors["val_ev_ebitda"]= winsorize_zscore(-raw["ev_ebitda"].dropna().reindex(raw.index))
factors["VALUE"] = factors[["val_pe", "val_pb", "val_ev_ebitda"]].mean(axis=1)

# ── QUALITY FACTOR ────────────────────────────────────────────────────────────
factors["q_roe"]       = winsorize_zscore(raw["roe"].dropna().reindex(raw.index))
factors["q_roa"]       = winsorize_zscore(raw["roa"].dropna().reindex(raw.index))
factors["q_net_margin"]= winsorize_zscore(raw["net_margin"].dropna().reindex(raw.index))
factors["q_curr_ratio"]= winsorize_zscore(raw["current_ratio"].dropna().reindex(raw.index))
factors["q_low_debt"]  = winsorize_zscore(-raw["debt_to_equity"].dropna().reindex(raw.index))
factors["QUALITY"] = factors[["q_roe", "q_roa", "q_net_margin", "q_curr_ratio", "q_low_debt"]].mean(axis=1)

# ── MOMENTUM FACTOR ───────────────────────────────────────────────────────────
factors["mom_12_1"]      = winsorize_zscore(raw["momentum_12_1"].dropna().reindex(raw.index))
factors["mom_sharpe"]    = winsorize_zscore(raw["sharpe_proxy"].dropna().reindex(raw.index))
factors["mom_reversal"]  = winsorize_zscore(-raw["reversal_1m"].dropna().reindex(raw.index))  # contrarian tilt
factors["MOMENTUM"] = factors[["mom_12_1", "mom_sharpe", "mom_reversal"]].mean(axis=1)

# ── LOW VOLATILITY FACTOR ─────────────────────────────────────────────────────
factors["lv_vol"]  = winsorize_zscore(-raw["vol_90d"].dropna().reindex(raw.index))
factors["lv_mdd"]  = winsorize_zscore(-raw["max_drawdown"].dropna().reindex(raw.index))
factors["LOW_VOL"] = factors[["lv_vol", "lv_mdd"]].mean(axis=1)

# ── GROWTH FACTOR ─────────────────────────────────────────────────────────────
factors["g_rev"]  = winsorize_zscore(raw["revenue_growth_yoy"].dropna().reindex(raw.index))
factors["g_earn"] = winsorize_zscore(raw["earnings_growth_yoy"].dropna().reindex(raw.index))
factors["GROWTH"] = factors[["g_rev", "g_earn"]].mean(axis=1)

# ── COMPOSITE SCORE ───────────────────────────────────────────────────────────
FACTOR_COLS   = ["VALUE", "QUALITY", "MOMENTUM", "LOW_VOL", "GROWTH"]
FACTOR_WEIGHTS = {"VALUE": 0.25, "QUALITY": 0.25, "MOMENTUM": 0.20, "LOW_VOL": 0.15, "GROWTH": 0.15}

factors["COMPOSITE"] = sum(
    factors[f] * w for f, w in FACTOR_WEIGHTS.items()
)

# Percentile rank (0–100)
factors["COMPOSITE_RANK"] = factors["COMPOSITE"].rank(pct=True) * 100

# Attach metadata
factors = factors.join(raw[["sector", "name"]], how="left")
factors = factors.dropna(subset=["COMPOSITE"])
factors = factors.sort_values("COMPOSITE_RANK", ascending=False)

print(f"Factor matrix built: {factors.shape}")
factors[FACTOR_COLS + ["COMPOSITE", "COMPOSITE_RANK"]].describe().round(3)

## 4. Screener Output — Top 30 Stocks

In [ ]:
# ── Top-30 Screener Table ─────────────────────────────────────────────────────
top30 = factors.head(30).copy()

display_cols = ["name", "sector"] + FACTOR_COLS + ["COMPOSITE", "COMPOSITE_RANK"]
top30_display = top30[display_cols].copy()
top30_display.index.name = "Ticker"

def color_score(val):
    """Green-to-red gradient for factor scores."""
    try:
        norm = plt.Normalize(-2, 2)
        cmap = plt.cm.RdYlGn
        rgba = cmap(norm(float(val)))
        r, g, b = int(rgba[0]*255), int(rgba[1]*255), int(rgba[2]*255)
        return f"background-color: rgb({r},{g},{b}); color: black"
    except Exception:
        return ""

styled = (
    top30_display.style
    .applymap(color_score, subset=FACTOR_COLS + ["COMPOSITE"])
    .format({col: "{:.3f}" for col in FACTOR_COLS + ["COMPOSITE"]})
    .format({"COMPOSITE_RANK": "{:.1f}"})
    .set_caption("Top 30 Stocks by Composite Factor Score")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "16px"), ("font-weight", "bold"), ("text-align", "left")]},
        {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("font-size", "12px")]},
    ])
)

styled

## 5. Factor Correlation Heatmap
Examines how much overlap exists between the five factor signals — low inter-factor correlation means better diversification.

In [ ]:
# ── Factor Correlation Heatmap ────────────────────────────────────────────────
corr = factors[FACTOR_COLS].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax,
    square=True,
    cbar_kws={"shrink": 0.8},
)
ax.set_title("Factor-to-Factor Correlation Matrix", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("factor_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Composite Score Distribution by Sector

In [ ]:
# ── Sector Composite Scores — Box Plot ───────────────────────────────────────
sector_order = (
    factors.groupby("sector")["COMPOSITE"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.box(
    factors.dropna(subset=["sector"]),
    x="sector",
    y="COMPOSITE",
    color="sector",
    category_orders={"sector": sector_order},
    title="Composite Factor Score Distribution by Sector",
    labels={"COMPOSITE": "Composite Z-Score", "sector": "GICS Sector"},
    height=520,
    template="plotly_dark",
)
fig.update_layout(
    showlegend=False,
    title_font_size=16,
    xaxis_tickangle=-35,
)
fig.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.4, annotation_text="Market Mean")
fig.show()

## 7. Top 30 — Factor Profile Radar Chart
Visualises the five-factor fingerprint for the top 10 composite names.

In [ ]:
# ── Radar Chart: Top 10 Factor Profiles ──────────────────────────────────────
top10 = factors.head(10)
categories = FACTOR_COLS + [FACTOR_COLS[0]]  # close the polygon

fig = go.Figure()

colorscale = px.colors.qualitative.Plotly

for i, (tkr, row) in enumerate(top10.iterrows()):
    vals = [row[f] for f in FACTOR_COLS]
    # Normalize to 0-1 for radar readability
    vals_norm = [(v - factors[f].min()) / (factors[f].max() - factors[f].min() + 1e-9) for f, v in zip(FACTOR_COLS, vals)]
    vals_norm += [vals_norm[0]]
    fig.add_trace(go.Scatterpolar(
        r=vals_norm,
        theta=categories,
        fill="toself",
        name=f"{tkr} ({row.get('name','')[:18]})",
        line_color=colorscale[i % len(colorscale)],
        opacity=0.7,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title=dict(text="Factor Profile — Top 10 Composite Stocks", font_size=16),
    template="plotly_dark",
    height=600,
    legend=dict(orientation="v", x=1.05),
)
fig.show()

## 8. Individual Factor Rankings — Top 15 per Factor

In [ ]:
# ── Bar Charts: Top 15 per Factor ────────────────────────────────────────────
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Top 15 by {f}" for f in FACTOR_COLS] + [""],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

positions = [(1,1),(1,2),(2,1),(2,2),(3,1)]

for (r, c), factor in zip(positions, FACTOR_COLS):
    top15_f = factors.nlargest(15, factor)[["name", factor, "sector"]].copy()
    top15_f["label"] = top15_f.index  # ticker
    fig.add_trace(
        go.Bar(
            x=top15_f["label"],
            y=top15_f[factor],
            marker_color=top15_f[factor],
            marker_colorscale="RdYlGn",
            showlegend=False,
            name=factor,
            hovertemplate="<b>%{x}</b><br>Score: %{y:.3f}<extra></extra>",
        ),
        row=r, col=c,
    )
    fig.update_xaxes(tickangle=-45, row=r, col=c)

fig.update_layout(
    title="Top 15 Stocks per Factor Signal",
    height=950,
    template="plotly_dark",
    title_font_size=16,
)
fig.show()

## 9. Momentum vs. Value Scatter — Bubble = Market Cap, Color = Composite Rank

In [ ]:
# ── Scatter: Momentum vs. Value ───────────────────────────────────────────────
plot_df = factors.dropna(subset=["VALUE", "MOMENTUM", "COMPOSITE_RANK", "sector"]).copy()
plot_df["market_cap"] = raw.reindex(plot_df.index)["market_cap"].fillna(1e10)
plot_df["label"] = plot_df.index

fig = px.scatter(
    plot_df,
    x="VALUE",
    y="MOMENTUM",
    color="COMPOSITE_RANK",
    size=plot_df["market_cap"].clip(1e8).apply(np.log10),
    hover_name="label",
    hover_data={"name": True, "sector": True, "COMPOSITE_RANK": ":.1f"},
    color_continuous_scale="RdYlGn",
    title="Value vs. Momentum Factor Scores (bubble = log market cap, color = composite rank)",
    labels={"VALUE": "Value Z-Score", "MOMENTUM": "Momentum Z-Score", "COMPOSITE_RANK": "Composite Rank (%)"},
    template="plotly_dark",
    height=600,
    opacity=0.75,
)

# Annotate top 10
top10_tickers = factors.head(10).index.tolist()
for tkr in top10_tickers:
    if tkr in plot_df.index:
        row = plot_df.loc[tkr]
        fig.add_annotation(
            x=row["VALUE"], y=row["MOMENTUM"],
            text=tkr, showarrow=True, arrowhead=2,
            font=dict(size=10, color="white"), arrowcolor="white",
            ax=20, ay=-20,
        )

fig.add_vline(x=0, line_dash="dot", line_color="grey", opacity=0.5)
fig.add_hline(y=0, line_dash="dot", line_color="grey", opacity=0.5)
fig.show()

## 10. Sector-Level Factor Averages — Heatmap
Which sectors are richest in which factors?

In [ ]:
# ── Sector Factor Heatmap ─────────────────────────────────────────────────────
sector_factors = (
    factors.dropna(subset=["sector"])
    .groupby("sector")[FACTOR_COLS]   # only group the 5 named factor cols
    .mean()
    .sort_values("VALUE", ascending=False)  # sort by VALUE, not COMPOSITE
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    sector_factors,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"shrink": 0.6},
)
ax.set_title("Avg Factor Score by GICS Sector", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("sector_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Export Results
Save the full screener output and top-30 to CSV.

In [ ]:
# ── Export to CSV ─────────────────────────────────────────────────────────────
export_cols = ["name", "sector"] + FACTOR_COLS + ["COMPOSITE", "COMPOSITE_RANK"]

all_scores_out = factors[export_cols].copy()
all_scores_out.index.name = "ticker"
all_scores_out.to_csv("multifactor_screener_all.csv")

top30[export_cols].to_csv("multifactor_screener_top30.csv")

print("✅ Exported:")
print("   multifactor_screener_all.csv  — full universe rankings")
print("   multifactor_screener_top30.csv — top 30 composite names")
print(f"\nRun timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n── TOP 10 COMPOSITE NAMES ──")
print(factors.head(10)[["name", "sector", "COMPOSITE", "COMPOSITE_RANK"]].to_string())

## Summary

This notebook builds a **five-factor quantitative equity screener** across the S&P 500 using live data from Yahoo Finance. It constructs cross-sectionally Z-scored signals for **Value** (P/E, P/B, EV/EBITDA), **Quality** (ROE, ROA, margins, leverage), **Momentum** (12-1 month price return, Sharpe proxy), **Low Volatility** (realized vol, max drawdown), and **Growth** (revenue and earnings YoY). Signals are winsorized to control outliers, combined into a weighted composite score, and ranked percentile-wise across the universe. Results are visualized via a styled factor heatmap, sector box plots, radar profiles, and an interactive Value-vs-Momentum scatter. Final outputs are exported as CSVs ready for portfolio construction.

---

**Resume bullet:**  
> Built a live multi-factor equity screener in Python (yfinance, pandas, Plotly) that ranks S&P 500 stocks across Value, Quality, Momentum, Low Volatility, and Growth signals using cross-sectional Z-scoring and percentile composite ranking, with interactive visualizations and CSV export.